# Fine-tuning Phi-3 avec LoRA (test local)

Adapté de l'article Hugging Face : https://huggingface.co/blog/dvgodoy/fine-tuning-llm-hugging-face

> `bitsandbytes` ne s'installe pas sur macOS : la quantization 4 bits (`BitsAndBytesConfig`, `prepare_model_for_kbit_training`) n'est activée que si CUDA est disponible.

In [ ]:
import json
import sys

import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

sys.path.append("..")  # pour importer src.constants depuis notebooks/

USE_CUDA = torch.cuda.is_available()
DEVICE = "cuda:0" if USE_CUDA else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device : {DEVICE}")

## 1. Hyper-paramètres

Isolés dans un dictionnaire pour pouvoir les logger avec les métriques / comme paramètres du composant.

In [ ]:
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

HYPERPARAMS = {
    # LoRA
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "lora_target_modules": ["o_proj", "qkv_proj", "gate_up_proj", "down_proj"],
    # Entraînement
    "num_train_epochs": 5,
    "learning_rate": 3e-4,
    "per_device_train_batch_size": 4,
    "max_seq_length": 512,
    "seed": 42,
}

## 2. Configurations

In [ ]:
lora_config = LoraConfig(
    r=HYPERPARAMS["lora_r"],
    lora_alpha=HYPERPARAMS["lora_alpha"],
    lora_dropout=HYPERPARAMS["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=HYPERPARAMS["lora_target_modules"],
)

if USE_CUDA:
    from transformers import BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float32,
    )
else:
    bnb_config = None  # pas de bitsandbytes sur macOS

sft_config = SFTConfig(
    output_dir="./phi3-mini-sustained-adapter",
    num_train_epochs=HYPERPARAMS["num_train_epochs"],
    learning_rate=HYPERPARAMS["learning_rate"],
    per_device_train_batch_size=HYPERPARAMS["per_device_train_batch_size"],
    auto_find_batch_size=True,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_seq_length=HYPERPARAMS["max_seq_length"],
    packing=True,
    # paged_adamw_8bit nécessite bitsandbytes
    optim="paged_adamw_8bit" if USE_CUDA else "adamw_torch",
    logging_steps=10,
    eval_strategy="epoch",
    # Métriques pour TensorBoard (dans le composant : logging_dir=metrics.path)
    report_to=["tensorboard"],
    logging_dir="./logs",
    seed=HYPERPARAMS["seed"],
)
lora_config

## 3. Chargement du modèle et préparation pour l'entraînement

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map=DEVICE,
    # Sans quantization, bfloat16 divise par 2 la mémoire (~7.6 Go au lieu de ~15 Go)
    torch_dtype=None if USE_CUDA else torch.bfloat16,
)

if USE_CUDA:
    model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id
tokenizer.padding_side = "right"

# On n'entraîne qu'une petite fraction des paramètres
model.print_trainable_parameters()

In [ ]:
# Les adaptateurs LoRA (lora_A : d_in x r, lora_B : r x d_out) sur une couche d'attention
print(model.base_model.model.model.layers[0].self_attn)

## 4. Chargement des datasets d'entraînement et de validation

Fichiers créés par `uv run python -m scripts.process_data_locally`, ou URI GCS des artifacts `train_output_path` et `validation_output_path` de la pipeline (visibles dans la console Vertex Pipelines).

In [ ]:
TRAIN_DATA_URI = "../data/train_sustained.csv"  # ou "gs://<bucket>/pipeline_root/.../train_output_path"
VALIDATION_DATA_URI = "../data/validation_sustained.csv"  # ou ".../validation_output_path"


def load_messages(uri: str) -> Dataset:
    df = pd.read_csv(uri)
    # Les messages ont été sérialisés en JSON par le composant de transformation
    return Dataset.from_dict({"messages": [json.loads(m) for m in df["messages"]]})


train_split = load_messages(TRAIN_DATA_URI)
eval_split = load_messages(VALIDATION_DATA_URI)

print(f"{len(train_split)} train, {len(eval_split)} validation")
train_split[0]["messages"]

In [ ]:
# Vérification du format après application du chat template
print(tokenizer.apply_chat_template(train_split[0]["messages"], tokenize=False))

## 5. Test de lancement de l'entraînement

En local on limite à quelques steps sur un sous-ensemble : le vrai entraînement tournera sur GPU dans la pipeline.

In [ ]:
sft_config.max_steps = 2  # test rapide uniquement

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=train_split.select(range(16)),
    eval_dataset=eval_split.select(range(4)),
)
trainer.train()